# **Data Wrangling and Preprocessing** #

Project Title: Identification of Significant Severe Weather Regimes Using Self-Organizing Maps

Authors: Kelvin Hawthorne and Lexi Gutzwiler

Northern Illinois University Department of Earth, Atmosphere, and Environment

**NOTEBOOK 1 FOR PROJECT METHODS

## **Environment Creation** ##
Since we are working in a python3 environment, we first need to setup our environment by pip installing what may not already exist

In [14]:
pip install xarray numpy pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## **Package Imports** ##
After we have pip installed the necessary packages, we now can import all the packages we will use over the course of opening the dataset

In [1]:
import matplotlib.pyplot as plt
from tqdm import tqdm #tqdm is a progress bar that will help follow the progress of opening such large data to ensure everything is working properly
import xarray as xr 
import pandas as pd
import numpy as np
import datetime
import glob
import os
import re

## **Path Specification** ##

In [ ]:
base_dir = "/glade/campaign/ncar/USGS_Water/CONUS404_PGW" #This is the data we are opening
out_dir = "/glade/derecho/scratch/khawthorne/485 project data" #This is where the final .nc file will live once created
geo_path = "/glade/derecho/scratch/khawthorne/CONUS404_geography.nc" #This is the geography file that specifies lat/long for the dataset to be plotted

os.makedirs(out_dir, exist_ok=True)

## **Open Dataset Preprocess** ##

### **Geography** ###
This geography file specifies our lat/longs so we can plot data geographically later

In [ ]:
geo = xr.open_dataset(geo_path).squeeze() #We are first opening the geography file

### **Temporal Chunking** ###
Since we are only working on a certain period, we will chunk the data only by the years we will use. This is more computationally efficient than opening the entire 43 year dataset.

In [ ]:
chunks = [
    list(range(2008, 2013)),  # 2008-2012
    list(range(2013, 2018)),  # 2013-2017
    list(range(2018, 2023)),  # 2018-2022
    list(range(2023, 2025)),  # 2023-2024
]

### **Date/Time Formatting** ###
CONUS404 data is output in minutes since initialization. This is the case for a lot of raw wrfout type files. Work needs to be done to convert that accordingly using datetime features.

In [ ]:
def get_file_time(path): #This helper function allows us to pull the correct UTC date/time from the file name and apply it to the file time in the new dataset
    m = re.search(
        r"wrf3d_d01_(\d{4}-\d{2}-\d{2}_\d{2}:\d{2}:\d{2})",
        os.path.basename(path)
    )
    if m:
        return pd.to_datetime(m.group(1), format="%Y-%m-%d_%H:%M:%S")
    return pd.NaT

### **File Preprocessing** ###
These steps are further preprocessing of the original dataset to make the opening and working with the data less computationally expensive, quicker, and more efficient.

In [ ]:
#Variable Selection
#To further minimze the computation expense of opening and saving this data, we can subset the dataset by only variables we will use for our project
def preprocess(ds):
    ds = ds[["Z", "P", "TK"]]

    # attach full geography as coordinates
    ds = ds.assign_coords(
        XLAT=(("south_north", "west_east"), geo["XLAT"].values),
        XLONG=(("south_north", "west_east"), geo["XLONG"].values),
    )

    # convert longitude to [-180, 180] if needed
    lon = xr.where(ds["XLONG"] > 180, ds["XLONG"] - 360, ds["XLONG"])
    ds = ds.assign_coords(XLONG=lon)

    # keep only points east of -107 longitude
    ds = ds.where(ds["XLONG"] >= -107, drop=True)

    # assign time from filename
    src = ds.encoding.get("source", "")
    file_time = get_file_time(src)
    ds = ds.assign_coords(time=("Time", [file_time]))
    ds = ds.swap_dims({"Time": "time"})

    return ds

### **Data Processing and Saving** ###
These steps are all to take the specifications from above and apply them to the dataset and output a new netcdf file that we can open that has all of our specifications.

In [ ]:
for wy_group in tqdm(chunks, desc="Chunk groups"):
    file_list = []

    for wy in tqdm(wy_group, desc=f"Collecting WY{wy_group[0]}-{wy_group[-1]}", leave=False):
        pattern = os.path.join(base_dir, f"WY{wy}", "wrf3d_d01_*")
        file_list.extend(sorted(glob.glob(pattern)))

    if not file_list:
        print(f"No files found for chunk WY{wy_group[0]}-WY{wy_group[-1]}")
        continue

    print(f"Opening WY{wy_group[0]}-WY{wy_group[-1]}: {len(file_list)} files")

    ds_chunk = xr.open_mfdataset(
        file_list,
        preprocess=preprocess,
        combine="nested",
        concat_dim="time",
        parallel=False,
        engine="netcdf4",
        chunks={"time": 1}
    )

    ds_chunk = ds_chunk.sortby("time")

    out_file = os.path.join(
        out_dir,
        f"wrf3d_ECONUS_Z_P_TK_WY{wy_group[0]}_WY{wy_group[-1]}.nc"
    )

    print(f"Saving {out_file}")
    ds_chunk.to_netcdf(out_file)
    ds_chunk.close()

print("Done.")